# VMF Mechanism

Implementation of the von Mises-Fisher distribution.

This file is based on the blog post by Whittenbury [1]. In fact, the numpy code is directly copied from
the blog post. We simply translated the code to PyTorch in order to work with tensors. Thereby,
we tried to keep the code as close as possible to the original numpy code.


[1] D. Whittenbury, “DS-2: sampling and visualising the von mises-fisher distribution in p dimensions,”
dlwhittenbury.com. Accessed: Nov. 18, 2024. [Online]. Available:
https://dlwhittenbury.github.io/ds-2-sampling-and-visualising-the-von-mises-fisher-distribution-in-p-dimensions.html

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from matplotlib import pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

from privtrajgen.privacy.vmf import *
from privtrajgen.utils import printc

# Formating
fs = 8
figsize = (4, 4)

## SPS Version

In [ ]:
from scipy import stats as sps

samples = sps.vonmises_fisher.rvs(mu=[1, 0, 0], kappa=1, size=10)
print(samples.shape)

## Sampling the uniform distribution on a $p$-dimensional hyper-sphere

In [ ]:
def plot_uniform_hypersphere_2d(data: np.ndarray):
    # Scatter plot of samples
    plt.figure(figsize=figsize)
    plt.scatter(data[:, 0], data[:, 1])

    # Make aspect square
    ax = plt.gca()
    # ax.set_aspect('equal')

    # Labels
    ax.set_xlabel('x', fontsize=fs)
    ax.set_ylabel('y', fontsize=fs)


def plot_uniform_hypersphere_3d(data: np.ndarray):
    plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d')
    ax.scatter(data[:, 0], data[:, 1], data[:, 2], s=5, c='mediumvioletred')

    # Labels
    ax.set_xlabel('x', fontsize=fs)
    ax.set_ylabel('y', fontsize=fs)
    ax.set_zlabel('z', fontsize=fs)

    # Limits
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_zlim(-1.5, 1.5)

    # View (use if inline plots are used)
    ax.view_init(20, 50)

In [ ]:
# Numpy implementation
data = rand_uniform_hypersphere_np(N=50, p=2)
printc("Data Shape (2D): ", data.shape)
plot_uniform_hypersphere_2d(data)

data = rand_uniform_hypersphere_np(N=1000, p=3)
printc("Data Shape (3D): ", data.shape)
plot_uniform_hypersphere_3d(data)

In [ ]:
# PyTorch Version
data = rand_uniform_hypersphere(N=50, p=2)
printc("Data Shape (2D): ", data.shape)
plot_uniform_hypersphere_2d(data.numpy())

data = rand_uniform_hypersphere(N=1000, p=3)
printc("Data Shape (3D): ", data.shape)
plot_uniform_hypersphere_3d(data.numpy())

## Sampling the marginal distribution of  using $t$-rejections sampling

In [ ]:
kappa = 1.0 / 2  # Epsilon / 2
p = 3
N = 50

In [ ]:
# Numpy implementation
data = rand_t_marginal_np(N=N, p=p, kappa=kappa)
printc("Data Shape (3D): ", data.shape)
printc("Data Type: ", type(data))
printc("Data Norm:", np.linalg.norm(data))
# PyTorch Version
data = rand_t_marginal(N=N, p=p, kappa=kappa)
printc("Data Shape (3D): ", data.shape)
printc("Data Type: ", type(data))
printc("Data Norm:", torch.norm(data))
# PyTorch Class Version
cls = RandTMarginal(kappa=kappa, p=p)
data = cls.sample(N=N)
printc("Data Shape (3D): ", data.shape)
printc("Data Type: ", type(data))
printc("Data Norm:", torch.norm(data))

In [ ]:
# Compare speed of Class and Functional implementation
# Pre-generate the class
cls = RandTMarginal(kappa=kappa, p=p)

In [ ]:
%%timeit -n 100 -r 3
data = rand_t_marginal(N=N, p=p, kappa=kappa)

In [ ]:
%%timeit -n 100 -r 3
data = cls.sample(N=N)

## rand_von_mises_fisher

In [ ]:
mu = np.random.randn(8)
mu = mu / np.linalg.norm(mu)
mu = torch.from_numpy(mu).float()
res = rand_von_mises_fisher(mu, kappa, 1)
print(res.shape)

## Plot Scripts

In [ ]:
def plot_3d_scatter(data, ax=None, colour='red', sz=5, el=20, az=50, sph=True, sph_colour="gray", sph_alpha=0.03,
                    eq_line=True, pol_line=True, grd=False):
    """
        plot_3d_scatter()
        =================

        Plots 3D samples on the surface of a sphere.

        INPUT:

            * data (array of floats of shape (N,3)) - samples of a spherical distribution such as von Mises-Fisher.
            * ax (axes) - axes on which the plot is constructed.
            * colour (string) - colour of the scatter plot.
            * sz (float) - size of points.
            * el (float) - elevation angle of the plot.
            * az (float) - azimuthal angle of the plot.
            * sph (boolean) - whether or not to inclde a sphere.
            * sph_colour (string) - colour of the sphere if included.
            * sph_alpha (float) - the opacity/alpha value of the sphere.
            * eq_line (boolean) - whether or not to include an equatorial line.
            * pol_line (boolean) - whether or not to include a polar line.
            * grd (boolean) - whether or not to include a grid.

        OUTPUT:

            * ax (axes) - axes on which the plot is contructed.
            * Plot of 3D samples on the surface of a sphere.

    """

    # The polar axis
    if ax is None:
        ax = plt.axes(projection='3d')

    # Check that data is 3D (data should be Nx3)
    d = np.shape(data)[1]
    if d != 3:
        raise Exception("data should be of shape Nx3, i.e., each data point should be 3D.")

    ax.scatter(data[:, 0], data[:, 1], data[:, 2], s=sz, c=colour)
    ax.view_init(el, az)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_zlim(-1.5, 1.5)

    # Add a shaded unit sphere
    if sph:
        u, v = np.mgrid[0:2 * np.pi:30j, 0:np.pi:30j]
        x = np.cos(u) * np.sin(v)
        y = np.sin(u) * np.sin(v)
        z = np.cos(v)
        ax.plot_surface(x, y, z, color=sph_colour, alpha=sph_alpha)

    # Add an equatorial line
    if eq_line:
        # t = theta, p = phi
        eqt = np.linspace(0, 2 * np.pi, 50, endpoint=False)
        eqp = np.linspace(0, 2 * np.pi, 50, endpoint=False)
        eqx = 2 * np.sin(eqt) * np.cos(eqp)
        eqy = 2 * np.sin(eqt) * np.sin(eqp) - 1
        eqz = np.zeros(50)

        # Equator line
        ax.plot(eqx, eqy, eqz, color="k", lw=1)

    # Add a polar line
    if pol_line:
        # t = theta, p = phi
        eqt = np.linspace(0, 2 * np.pi, 50, endpoint=False)
        eqp = np.linspace(0, 2 * np.pi, 50, endpoint=False)
        eqx = 2 * np.sin(eqt) * np.cos(eqp)
        eqy = 2 * np.sin(eqt) * np.sin(eqp) - 1
        eqz = np.zeros(50)

        # Polar line
        ax.plot(eqx, eqz, eqy, color="k", lw=1)

    # Draw a centre point
    ax.scatter([0], [0], [0], color="k", s=sz)

    # Turn off grid
    ax.grid(grd)

    # Ticks
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    ax.set_zticks([-1, 0, 1])

    return ax


# Drawing a fancy vector see Ref. [7]
class Arrow3D(FancyArrowPatch):

    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.figure.gca().get_proj())
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        super().draw(renderer)

    def do_3d_projection(self):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.figure.gca().get_proj())
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return zs[0]


def plot_arrow(point, ax, colour="red", lw=3):
    """
        plot_arrow(point,ax,colour="red")
        ==============================

        Plots a 3D arrow on the axes ax from the origin to the point mu.

        INPUT:

            * point (array of floats of shape (3,1)) - a 3D point.
            * ax (axes) - axes on which the plot is constructed.
            * colour (string) - colour of the arrow.
            * lw (float) - line width of the arrow.

    """

    # Can use quiver for a simple arrow
    #ax.quiver(0,0,0,point[0],point[1],point[2],length=1.0,color=colour,pivot="tail")

    # Fancy arrow
    a = Arrow3D([0, point[0]], [0, point[1]], [0, point[2]], mutation_scale=20, lw=lw, arrowstyle="-|>", color=colour)
    ax.add_artist(a)

    return ax

## Plot Numpy Version

In [ ]:
# All sets have the same number of data points
Nsim = 500

# Set 1
mu1 = [1, 1, 0]
mu1 = mu1 / np.linalg.norm(mu1)
kappa1 = 50
data1 = rand_von_mises_fisher_np(mu1, kappa=kappa1, N=Nsim)

# Set 2
mu2 = [0, 0, 1]
mu2 = mu2 / np.linalg.norm(mu2)
kappa2 = 20
data2 = rand_von_mises_fisher_np(mu2, kappa=kappa2, N=Nsim)

# Set 3
mu3 = [0, 0, -1]
mu3 = mu3 / np.linalg.norm(mu3)
kappa3 = 20
data3 = rand_von_mises_fisher_np(mu3, kappa=kappa3, N=Nsim)

# Set 4
mu4 = [-10, 0, -1]
mu4 = mu4 / np.linalg.norm(mu4)
kappa4 = 200
data4 = rand_von_mises_fisher_np(mu4, kappa=kappa4, N=Nsim)

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')

# Set 1
plot_3d_scatter(data1, ax)
plot_arrow(mu1, ax, colour="red")

# Set 2
plot_3d_scatter(data2, ax, colour='orange')
plot_arrow(mu2, ax, colour="orange")

# Set 3
plot_3d_scatter(data3, ax, colour='blue')
plot_arrow(mu3, ax, colour="blue")

# Set 4
plot_3d_scatter(data4, ax, colour='green')
plot_arrow(mu4, ax, colour="green")

# Labels
ax.set_xlabel('x', fontsize=fs)
ax.set_ylabel('y', fontsize=fs)
ax.set_zlabel('z', fontsize=fs)

# Viewing angle
ax.view_init(20, 120)

# Save it
# plt.savefig("vmf-sphere-p-3-scatter-w-arrows.png")
plt.show()

## Plot PyTorch Version

In [ ]:
# All sets have the same number of data points
Nsim = 500

# Set 1
mu1 = torch.tensor([1, 1, 0], dtype=torch.float)
mu1 = mu1 / torch.norm(mu1)
kappa1 = 50
data1 = rand_von_mises_fisher(mu1, kappa=kappa1, N=Nsim)
printc("Data Shape: ", data1.shape)

# Set 2
mu2 = torch.tensor([0, 0, 1], dtype=torch.float)
mu2 = mu2 / torch.norm(mu2)
kappa2 = 20
data2 = rand_von_mises_fisher(mu2, kappa=kappa2, N=Nsim)

# Set 3
mu3 = torch.tensor([0, 0, -1], dtype=torch.float)
mu3 = mu3 / torch.norm(mu3)
kappa3 = 20
data3 = rand_von_mises_fisher(mu3, kappa=kappa3, N=Nsim)

# Set 4
mu4 = torch.tensor([-10, 0, -1], dtype=torch.float)
mu4 = mu4 / torch.norm(mu4)
kappa4 = 200
data4 = rand_von_mises_fisher(mu4, kappa=kappa4, N=Nsim)

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')

# Set 1
plot_3d_scatter(data1, ax)
plot_arrow(mu1, ax, colour="red")

# Set 2
plot_3d_scatter(data2, ax, colour='orange')
plot_arrow(mu2, ax, colour="orange")

# Set 3
plot_3d_scatter(data3, ax, colour='blue')
plot_arrow(mu3, ax, colour="blue")

# Set 4
plot_3d_scatter(data4, ax, colour='green')
plot_arrow(mu4, ax, colour="green")

# Labels
ax.set_xlabel('x', fontsize=fs)
ax.set_ylabel('y', fontsize=fs)
ax.set_zlabel('z', fontsize=fs)

# Viewing angle
ax.view_init(20, 120)

# Save it
# plt.savefig("vmf-sphere-p-3-scatter-w-arrows.png")
plt.show()

## Plot Epsilon

In [ ]:
# Values
epsilon = 10.0
sigma = epsilon
dim = 3

VMF = RandVonMisesFisher(kappa=sigma, p=dim)

# All sets have the same number of data points
Nsim = 500

# Set 1
mu1 = torch.tensor([1, 1, 0], dtype=torch.float)
mu1 = mu1 / torch.norm(mu1)
# Add a batch dimension = Nsim
mu1_b = mu1.repeat(Nsim, 1)
printc("mu1 Shape: ", mu1.shape)
data1 = VMF(mu1_b)
printc("Data Shape: ", data1.shape)

# Set 2
mu2 = torch.tensor([0, 0, 1], dtype=torch.float)
mu2 = mu2 / torch.norm(mu2)
mu2_b = mu2.repeat(Nsim, 1)  # Add a batch dimension = Nsim
data2 = VMF(mu2_b)

# Set 3
mu3 = torch.tensor([0, 0, -1], dtype=torch.float)
mu3 = mu3 / torch.norm(mu3)
mu3_b = mu3.repeat(Nsim, 1)  # Add a batch dimension = Nsim
data3 = VMF(mu3_b)

# Set 4
mu4 = torch.tensor([-10, 0, -1], dtype=torch.float)
mu4 = mu4 / torch.norm(mu4)
mu4_b = mu4.repeat(Nsim, 1)  # Add a batch dimension = Nsim
data4 = VMF(mu4_b)

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')

# Set 1
plot_3d_scatter(data1, ax)
plot_arrow(mu1, ax, colour="red")

# Set 2
plot_3d_scatter(data2, ax, colour='orange')
plot_arrow(mu2, ax, colour="orange")

# Set 3
plot_3d_scatter(data3, ax, colour='blue')
plot_arrow(mu3, ax, colour="blue")

# Set 4
plot_3d_scatter(data4, ax, colour='green')
plot_arrow(mu4, ax, colour="green")

# Labels
ax.set_xlabel('x', fontsize=fs)
ax.set_ylabel('y', fontsize=fs)
ax.set_zlabel('z', fontsize=fs)

# Viewing angle
ax.view_init(20, 120)

# LaTeX title: "VMF mechanism for $\sigma = \varepsilon = 10$"
plt.title(r"VMF mechanism for $\sigma = \varepsilon = 10$", fontsize=15, y=1.0)

plt.tight_layout(pad=1.0)

# Save it
# plt.savefig("vmf-sphere-p-3-scatter-w-arrows.png")
plt.show()

## Plot for Thesis

In [ ]:
# Values
# epsilon = 10.0
sigma = 10.0
dim = 3

VMF = RandVonMisesFisher(kappa=sigma, p=dim)

# All sets have the same number of data points
Nsim = 250

# Set 1
mu1 = torch.tensor([1, 1, 0], dtype=torch.float)
mu1 = mu1 / torch.norm(mu1)
# Add a batch dimension = Nsim
mu1_b = mu1.repeat(Nsim, 1)
printc("mu1 Shape: ", mu1.shape)
data1 = VMF(mu1_b)
printc("Data Shape: ", data1.shape)

# Set 2
mu2 = torch.tensor([0, 0, 1], dtype=torch.float)
mu2 = mu2 / torch.norm(mu2)
mu2_b = mu2.repeat(Nsim, 1)  # Add a batch dimension = Nsim
data2 = VMF(mu2_b)

# Set 3
mu3 = torch.tensor([0, 0, -1], dtype=torch.float)
mu3 = mu3 / torch.norm(mu3)
mu3_b = mu3.repeat(Nsim, 1)  # Add a batch dimension = Nsim
data3 = VMF(mu3_b)

# Set 4
mu4 = torch.tensor([-10, 0, -1], dtype=torch.float)
mu4 = mu4 / torch.norm(mu4)
mu4_b = mu4.repeat(Nsim, 1)  # Add a batch dimension = Nsim
data4 = VMF(mu4_b)

In [ ]:
import os

from privtrajgen import config as conf

# Configuration
OUTPUT_DIR = os.path.join(conf.BASE_DIR, 'img/')
CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache/')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


def pt_to_cm(pt):
    return pt * 0.0352778


def cm_to_inch(cm):
    """Matplotlib figsize accepts inches"""
    return cm / 2.54


def pt_to_inch(pt):
    return cm_to_inch(pt_to_cm(pt))


# Thesis Dimensions (Note that matplotlib uses inches, use plot.cm to convert to cm)
LINEWIDTH_THESIS = 426.79135  # pt
# HALF_LINEWIDTH_THESIS = 7.3776451944  # in cm, equals 213.395675 pt
# Thesis font sizes
# The thesis used font 'LM Roman 10' with 10.95pt for captions and 10pt for sub-captions
# In the default matplotlib font this equals

FONT_SIZE = 9
# Update all relevant font size rcParams
plt.rcParams.update({
    'font.size': FONT_SIZE,  # Global font size
    'axes.titlesize': FONT_SIZE,  # Title font size
    'axes.labelsize': FONT_SIZE,  # X and Y axis labels
    'xtick.labelsize': FONT_SIZE - 2,  # X-axis tick labels
    'ytick.labelsize': FONT_SIZE - 2,  # Y-axis tick labels
    'legend.fontsize': FONT_SIZE,  # Legend font size
    'figure.titlesize': FONT_SIZE,  # Figure title font size
    'figure.labelsize': FONT_SIZE,  # Figure label font size
    'lines.markersize': 3,  # Adjust marker size
    'figure.subplot.wspace': 0.1,  # Reduce space between subfigures
    'figure.subplot.hspace': 0.1,  # Reduce vertical space to 0.1
})

fig_width = pt_to_inch(LINEWIDTH_THESIS)
aspect_ratio = 2
fig_height = fig_width / aspect_ratio

# colors = ['red', 'orange', 'blue', 'green']
# Get Tab10 colormap
cmap = plt.get_cmap('tab10')
colors = [cmap(i) for i in range(4)]

In [ ]:
# Values not in rcParams
OUTPUT_FORMAT = "png"
SZ = 1
EL = 10
AZ = 25
LW = 2

fig = plt.figure(figsize=(fig_width, fig_height))
ax = plt.axes(projection='3d')

# Set 1
plot_3d_scatter(data1, ax, colour=colors[0], sz=SZ, el=EL, az=AZ)
plot_arrow(mu1, ax, colour=colors[0], lw=LW)

# Set 2
plot_3d_scatter(data2, ax, colour=colors[1], sz=SZ, el=EL, az=AZ)
plot_arrow(mu2, ax, colour=colors[1], lw=LW)

# Set 3
plot_3d_scatter(data3, ax, colour=colors[2], sz=SZ, el=EL, az=AZ)
plot_arrow(mu3, ax, colour=colors[2], lw=LW)

# Set 4
plot_3d_scatter(data4, ax, colour=colors[3], sz=SZ, el=EL, az=AZ)
plot_arrow(mu4, ax, colour=colors[3], lw=LW)

# Labels
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')

# Viewing angle
ax.view_init(20, 120)

# LaTeX title: "VMF mechanism for $\sigma = \varepsilon = 10$"
plt.title(r"VMF mechanism for $\kappa = 10$", y=0.96)

# plt.tight_layout()
plt.subplots_adjust(left=0.0, right=1.0, top=1.03, bottom=0.03)

# Save it
plt.savefig(os.path.join(OUTPUT_DIR, "vmf_example." + 'pdf'), dpi=300, pad_inches=0.0)
plt.savefig(os.path.join(OUTPUT_DIR, "vmf_example." + 'png'), dpi=300, pad_inches=0.0)
plt.show()